# Programming in Java – Assignment 4 (CO4)
## Event-Driven GUI Design

This notebook prepares and compiles the Java source files for Tasks 1–6, records console-based verification tests, and packages the source code, SQL script, design notes, and test outputs.

**Student:** J. Subhaashree  
**Register Number:** 192424008  
**Course:** Programming in Java (CSA09)


## 1. Set up Java and create the assignment folders

In [1]:
import os, pathlib, subprocess, textwrap, zipfile, json, shutil, csv, time
from pathlib import Path

BASE = Path("/content/CSA09-192424008-Assignment4")
if BASE.exists():
    shutil.rmtree(BASE)
for folder in ["Task01","Task02","Task03","Task04","Task05","Task06","docs","sql","logs","data","screenshots"]:
    (BASE/folder).mkdir(parents=True, exist_ok=True)

print(subprocess.run(["bash","-lc","java -version && javac -version"],capture_output=True,text=True).stderr)
print("Created:", BASE)


openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)

Created: /content/CSA09-192424008-Assignment4


## 2. Task 1 – GUI design review and repaired fee-payment design

In [2]:
task1 = r"""
import java.awt.*;
import java.awt.event.*;
import javax.swing.*;
import java.sql.*;

class FeeService {
    private final String url, user, password;
    FeeService(String url, String user, String password) {
        this.url=url; this.user=user; this.password=password;
    }
    String pay(String regNo, String amountText) throws Exception {
        if (regNo == null || regNo.trim().isEmpty()) throw new IllegalArgumentException("Registration number is required");
        double amount;
        try { amount = Double.parseDouble(amountText.trim()); }
        catch (NumberFormatException ex) { throw new IllegalArgumentException("Enter a valid amount"); }
        if (!Double.isFinite(amount) || amount <= 0) throw new IllegalArgumentException("Amount must be greater than zero");
        String sql = "INSERT INTO fees(reg_no, amount) VALUES (?, ?)";
        try (Connection c=DriverManager.getConnection(url,user,password);
             PreparedStatement ps=c.prepareStatement(sql)) {
            ps.setString(1,regNo.trim()); ps.setDouble(2,amount); ps.executeUpdate();
        }
        return String.format("Payment of %.2f recorded for %s",amount,regNo.trim());
    }
}
public class FeeWindow extends JFrame {
    private final JTextField regNo=new JTextField(12), amount=new JTextField(10);
    private final JLabel status=new JLabel("Enter fee details");
    private final FeeService service=new FeeService(
        System.getenv().getOrDefault("FEE_DB_URL","jdbc:mysql://localhost:3306/campus"),
        System.getenv().getOrDefault("FEE_DB_USER","root"),
        System.getenv().getOrDefault("FEE_DB_PASSWORD",""));
    public FeeWindow() {
        super("Student Fee Payment");
        JPanel form=new JPanel(new GridLayout(2,2,8,8));
        form.add(new JLabel("Registration No.")); form.add(regNo);
        form.add(new JLabel("Amount")); form.add(amount);
        JButton pay=new JButton("Pay"), clear=new JButton("Clear"), receipt=new JButton("Receipt");
        JPanel actions=new JPanel(); actions.add(pay); actions.add(clear); actions.add(receipt);
        add(form,BorderLayout.NORTH); add(actions,BorderLayout.CENTER); add(status,BorderLayout.SOUTH);
        pay.addActionListener(e -> payFee());
        clear.addActionListener(e -> { regNo.setText(""); amount.setText(""); status.setText("Form cleared"); });
        receipt.addActionListener(e -> JOptionPane.showMessageDialog(this,status.getText(),"Payment Receipt",JOptionPane.INFORMATION_MESSAGE));
        addWindowListener(new WindowAdapter(){ @Override public void windowClosing(WindowEvent e){ dispose(); } });
        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE); setSize(420,220); setLocationRelativeTo(null);
    }
    private void payFee() {
        final String id=regNo.getText(), value=amount.getText();
        status.setText("Processing...");
        new SwingWorker<String,Void>() {
            protected String doInBackground() throws Exception { return service.pay(id,value); }
            protected void done() {
                try {
                    String message=get();
                    status.setText(message);
                    JOptionPane.showMessageDialog(FeeWindow.this,message,"Payment",JOptionPane.INFORMATION_MESSAGE);
                } catch (Exception ex) {
                    Throwable cause=ex.getCause()==null?ex:ex.getCause();
                    status.setText("Payment could not be completed");
                    JOptionPane.showMessageDialog(FeeWindow.this,cause.getMessage(),"Payment Error",JOptionPane.ERROR_MESSAGE);
                }
            }
        }.execute();
    }
    public static void main(String[] args) { SwingUtilities.invokeLater(() -> new FeeWindow().setVisible(true)); }
}
"""
(BASE/"Task01"/"FeeWindow.java").write_text(task1)
diagnosis = """Task 1.1 – Design deficiencies
1. One listener owns unrelated actions, coupling UI events and business logic; changes become difficult to isolate.
2. String values are compared with ==, which compares object references. Use "Pay".equals(e.getActionCommand()) or distinct listeners.
3. Empty, non-numeric, negative and non-finite amounts are not validated; parsing may throw NumberFormatException.
4. JDBC work runs on the event-dispatch thread (EDT), so the window freezes while the database responds.
5. Thread.sleep(3000) on the EDT freezes repainting and input for three seconds.
6. The Receipt button is not implemented as a receipt action; it falls into the clear branch.
7. SQL is built by concatenation, allowing SQL injection and quoting errors. Use PreparedStatement.
8. The connection and statement are not closed, risking resource leaks.
9. There is no window-closing policy or explicit resource lifecycle.
10. SQLException and InterruptedException are not handled or reported to the user.
11. No success/failure feedback is shown in the window; System.out is not useful feedback for the receptionist.

Task 1.2 – Repaired structure
FeeWindow (Swing view) collects fields and displays status/dialogs. FeeService validates and performs payment. The database operation uses PreparedStatement and try-with-resources. For a production application, call FeeService from SwingWorker.doInBackground() and update status in done(), keeping JDBC off the EDT.

Task 1.3 – Design defence (under 100 words)
The view only manages components and user interaction, while FeeService owns validation and payment rules. Each class has one responsibility, so validation can be tested without opening a window and the interface can change without rewriting database logic. PreparedStatement prevents SQL injection, and try-with-resources closes JDBC resources reliably. Separate listeners make each action explicit. A SwingWorker keeps slow database operations off the event-dispatch thread, preserving responsiveness.
"""
(BASE/"Task01"/"design_review.txt").write_text(diagnosis)
print("Task 1 source and design review written.")


Task 1 source and design review written.


## 3. Task 2 – Hospital outpatient appointment desk

In [3]:
task2 = r"""
import java.util.*;
class Doctor {
    final String id, name;
    Doctor(String id,String name){this.id=id;this.name=name;}
    public String toString(){return name;}
}
class AppointmentService {
    private final Map<String,List<Doctor>> doctorsByDepartment = new HashMap<>();
    AppointmentService(){
        doctorsByDepartment.put("Cardiology",Arrays.asList(new Doctor("C1","Dr Rao"),new Doctor("C2","Dr Iyer")));
        doctorsByDepartment.put("Orthopaedics",Arrays.asList(new Doctor("O1","Dr Kumar"),new Doctor("O2","Dr Das")));
        doctorsByDepartment.put("Paediatrics",Arrays.asList(new Doctor("P1","Dr Meena")));
    }
    List<Doctor> doctorsFor(String department){return doctorsByDepartment.getOrDefault(department,Collections.emptyList());}
    String book(String patientId,String department,String doctor,String slot){
        if(patientId==null||patientId.trim().isEmpty()) throw new IllegalArgumentException("Patient ID is required");
        if(department==null||department.isEmpty()||doctor==null||doctor.isEmpty()||slot==null||slot.isEmpty())
            throw new IllegalArgumentException("Select department, doctor and time slot");
        return "Appointment confirmed: "+patientId+" | "+department+" | "+doctor+" | "+slot;
    }
}
public class AppointmentDesk {
    public static void main(String[] args){
        AppointmentService s=new AppointmentService();
        System.out.println("Cardiology doctors: "+s.doctorsFor("Cardiology"));
        System.out.println(s.book("P1001","Cardiology","Dr Rao","10:30"));
        try{s.book("","Cardiology","Dr Rao","10:30");}
        catch(IllegalArgumentException e){System.out.println("Validation: "+e.getMessage());}
    }
}
"""
(BASE/"Task02"/"AppointmentDesk.java").write_text(task2)
notes = """Task 2.1 – Event model
Source | Event object | Listener | Action
Department JComboBox | ItemEvent | ItemListener | Replace doctor choices for selected department.
Doctor JComboBox | ItemEvent | ItemListener | Store selected doctor and validate selection.
Time-slot JComboBox | ItemEvent | ItemListener | Store selected slot.
Book JButton | ActionEvent | ActionListener | Validate fields, book appointment, show confirmation/error.
Window | WindowEvent | WindowListener/WindowAdapter | If booking is in progress, ask before closing; otherwise dispose.
Patient ID JTextField | ActionEvent | ActionListener (optional) | Move focus to department selection.

Task 2.2 – Component hierarchy
JFrame AppointmentDesk
└── JPanel (BorderLayout)
    ├── JPanel NORTH (GridLayout 4x2): patient ID JLabel/JTextField, department JLabel/JComboBox, doctor JLabel/JComboBox, slot JLabel/JComboBox
    ├── JPanel CENTER: instructions/status JLabel or appointment summary JTextArea
    └── JPanel SOUTH (FlowLayout): Book JButton, Clear JButton

Task 2.3 – Handler pseudocode
REFRESH_DOCTORS:
  department ← selected department
  doctorCombo.removeAllItems()
  FOR EACH doctor IN doctorsByDepartment.get(department)
      doctorCombo.addItem(doctor)
  END FOR
  IF doctorCombo.getItemCount() = 0 THEN disable Book
  ELSE select first doctor and enable Book
  END IF

BOOK_APPOINTMENT:
  IF patientID is blank THEN show validation error; RETURN
  IF department, doctor or slot is not selected THEN show validation error; RETURN
  bookingInProgress ← TRUE
  disable Book
  result ← AppointmentService.book(patientID, department, doctor, slot)
  show confirmation(result)
  bookingInProgress ← FALSE
  enable Book
  ON failure show error and set bookingInProgress ← FALSE

Task 2.4 – Event trace
main → SwingUtilities.invokeLater → construct JFrame → initialise components/listeners → pack/setSize → setVisible(true) → user event enters EDT → listener method executes → service method executes → status/dialog updates → windowClosing event → ask confirmation if bookingInProgress → windowClosed/dispose.
Applet comparison: init() initialises once; start() begins/restarts activity; paint() draws; stop() pauses activity; destroy() releases resources. A JFrame does not have this applet lifecycle; its construction, visibility and window events are managed by the application and AWT event queue.

Alternative and complexity:
With a preloaded Map, department lookup is expected O(1) plus O(r) to populate r doctors in the selected department; memory is O(D+M) for D departments and M total doctors. Fetching from a database on each change costs network/query latency plus O(r) result processing, and repeated queries add database load. It can reduce local cache memory to O(r) but introduces latency and failure handling on each change. Cache refresh/invalidation is needed if doctor data changes.
"""
(BASE/"Task02"/"design_and_pseudocode.txt").write_text(notes)
print("Task 2 source and design notes written.")


Task 2 source and design notes written.


## 4. Task 3 – Campus library search screen (AWT/Swing implementation)

In [4]:
task3 = r"""
import java.awt.*;
import java.awt.event.*;
import javax.swing.*;
import javax.swing.table.DefaultTableModel;

public class LibrarySearch extends JFrame {
    private final JTextField query=new JTextField(22);
    private final JComboBox<String> category=new JComboBox<>(new String[]{"All","Title","Author","Subject"});
    private final DefaultTableModel model=new DefaultTableModel(new Object[]{"Title","Author","Category","Availability"},0);
    private final JTable results=new JTable(model);
    private final JLabel status=new JLabel("Enter a search term.");
    public LibrarySearch(){
        super("Campus Library Search");
        setLayout(new BorderLayout(8,8));
        JPanel search=new JPanel(new FlowLayout(FlowLayout.LEFT));
        search.add(new JLabel("Search")); search.add(query); search.add(category);
        JButton searchButton=new JButton("Search"); search.add(searchButton);
        JPanel bottom=new JPanel(new FlowLayout(FlowLayout.CENTER));
        JButton prev=new JButton("Prev"), next=new JButton("Next"), reserve=new JButton("Reserve");
        bottom.add(prev); bottom.add(next); bottom.add(reserve);
        JPanel south=new JPanel(new BorderLayout()); south.add(bottom,BorderLayout.CENTER); south.add(status,BorderLayout.SOUTH);
        add(search,BorderLayout.NORTH); add(new JScrollPane(results),BorderLayout.CENTER); add(south,BorderLayout.SOUTH);
        searchButton.addActionListener(e -> searchBooks());
        query.addActionListener(e -> searchBooks());
        prev.addActionListener(e -> status.setText("Previous results requested"));
        next.addActionListener(e -> status.setText("Next results requested"));
        reserve.addActionListener(e -> status.setText("Select a result to reserve"));
        setDefaultCloseOperation(EXIT_ON_CLOSE); setMinimumSize(new Dimension(520,320)); setSize(760,480); setLocationRelativeTo(null);
    }
    private void searchBooks(){
        String q=query.getText().trim();
        if(q.isEmpty()){status.setText("Enter a search term.");return;}
        model.setRowCount(0);
        model.addRow(new Object[]{"Java Programming","Herbert Schildt","Computer Science","Available"});
        status.setText("Search complete for: "+q);
    }
    public static void main(String[] args){SwingUtilities.invokeLater(()->new LibrarySearch().setVisible(true));}
}
"""
(BASE/"Task03"/"LibrarySearch.java").write_text(task3)
(BASE/"Task03"/"layout_design.txt").write_text("""Task 3.1 – Layout choices
• JFrame uses BorderLayout: NORTH search row, CENTER scrollable results, SOUTH action/status region. CENTER receives remaining space and grows on resize.
• Search row uses FlowLayout(LEFT), keeping components at preferred sizes.
• Bottom action row uses FlowLayout(CENTER), keeping Prev/Next/Reserve at preferred sizes.
• SOUTH wrapper uses BorderLayout so action row stays CENTER and status stays SOUTH.
• JTable is placed in JScrollPane; the viewport expands with the CENTER region.

Task 3.2 – Nested structure
JFrame [BorderLayout]
├── Search JPanel [FlowLayout LEFT] – NORTH
├── JScrollPane(JTable) – CENTER
└── South JPanel [BorderLayout] – SOUTH
    ├── Actions JPanel [FlowLayout CENTER] – CENTER
    └── Status JLabel – SOUTH

Task 3.4 – Resize verification plan
Run at 520×320, 760×480 and 1100×700. At each size capture the whole window. Expected: results viewport expands/shrinks with the frame; search controls and bottom buttons retain preferred dimensions; status remains visible below the action row.
Alternative: GridBagLayout can express weighted CENTER growth but requires constraints and more setup. Absolute positioning uses fixed coordinates and does not adapt to resizing; controls overlap or leave gaps. Nested BorderLayout/FlowLayout layout work is approximately O(c) per layout pass for c components and uses O(p) additional panel/container objects. GridBagLayout also visits components/constraints (approximately O(c)), with more constraint state. Absolute positioning avoids constraint calculations but still has O(c) component placement and poor responsiveness. Number of containers is O(1) for the fixed nested design.
""")
print("Task 3 source and layout notes written.")


Task 3 source and layout notes written.


## 5. Task 4 – Event-handling forensics and completed DrawPad

In [5]:
task4 = r"""
import java.awt.*;
import java.awt.event.*;
import java.util.*;
import java.util.List;

public class DrawPad extends Frame {
    private final List<Point> points = new ArrayList<>();
    private final Label status = new Label("Clicks: 0");
    public DrawPad() {
        super("Draw Pad");
        setLayout(new BorderLayout());
        add(status,BorderLayout.SOUTH);
        addMouseListener(new MouseAdapter() {
            @Override public void mouseClicked(MouseEvent e) {
                points.add(e.getPoint());
                status.setText("Clicks: "+points.size());
                repaint();
            }
        });
        addWindowListener(new WindowAdapter() {
            @Override public void windowClosing(WindowEvent e) { dispose(); }
        });
        setSize(400,300);
    }
    @Override public void paint(Graphics g) {
        super.paint(g);
        for(Point p:points) g.fillOval(p.x-3,p.y-3,6,6);
    }
    public static void main(String[] args) { EventQueue.invokeLater(() -> new DrawPad().setVisible(true)); }
}
"""
(BASE/"Task04"/"DrawPad.java").write_text(task4)
notes="""Task 4.1 – Defects
The method is spelled mouseclicked, but MouseAdapter defines mouseClicked with a capital C. Java is case-sensitive, so the method does not override the listener callback and is never invoked. The label is never updated because the callback body never runs. Even if corrected, the code draws directly with getGraphics(), which is temporary and is not retained for future repaint operations.

Task 4.2 – Failing interaction
Click several times to draw dots, minimise the window, then restore it. The window system requests repainting; the previous direct getGraphics() drawings are not part of the component's persistent painting state, so the dots disappear.

Task 4.3 – Repair
The completed DrawPad stores each click as a Point, updates the label, calls repaint(), and redraws all stored points in paint(). Each dot is a 6×6 oval centred at the click coordinate.

Task 4.4 – Prevention
Adding @Override above the incorrectly spelled mouseclicked method makes the compiler report that the method does not override or implement a method from a supertype. This catches the typo before runtime.

Complete-code alternative: A BufferedImage can be created at the component's current width and height. Each click draws into the image using Graphics2D; paint() copies the image to the screen. Repainting then copies one image (O(W×H) pixel transfer in the straightforward model, often accelerated), instead of redrawing n points in O(n). The point-list design stores O(n) Point objects; the image design stores O(W×H) pixels (approximately 4WH bytes for ARGB). The image must be recreated/rescaled when the window dimensions change.
"""
(BASE/"Task04"/"forensics_and_analysis.txt").write_text(notes)
# create intentionally failing sample for compile-time @Override demonstration
(BASE/"Task04"/"MouseTypoCompileFail.java").write_text("""import java.awt.event.*;\nclass MouseTypoCompileFail extends MouseAdapter {\n    @Override public void mouseclicked(MouseEvent e) { }\n}\n""")
print("Task 4 source and notes written.")


Task 4 source and notes written.


## 6. Task 5 – Currency converter in four event-handling styles

In [6]:
common = r"""
import java.awt.*;
import java.awt.event.*;
import javax.swing.*;
import java.text.DecimalFormat;

"""
frame_impl = common + r"""
public class ConverterFrameListener extends JFrame implements ActionListener {
    JTextField amount=new JTextField(10); JComboBox<String> target=new JComboBox<>(new String[]{"USD","EUR"});
    JLabel result=new JLabel(" "); JButton button=new JButton("Convert");
    ConverterFrameListener(){super("Converter – Frame Listener");setLayout(new FlowLayout());add(new JLabel("INR"));add(amount);add(target);add(button);add(result);button.addActionListener(this);amount.addActionListener(this);addWindowListener(new WindowAdapter(){public void windowClosing(WindowEvent e){dispose();}});
        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE);pack();}
    public void actionPerformed(ActionEvent e){convert();}
    private void convert(){try{double v=Double.parseDouble(amount.getText().trim());if(v<0||!Double.isFinite(v))throw new NumberFormatException();String c=(String)target.getSelectedItem();String s=new DecimalFormat("0.00").format(v*(c.equals("USD")?0.012:0.011));result.setText(s+" "+c);JOptionPane.showMessageDialog(this,result.getText());}catch(NumberFormatException ex){result.setText("Invalid amount");JOptionPane.showMessageDialog(this,"Invalid amount","Error",JOptionPane.ERROR_MESSAGE);}}
    public static void main(String[]a){SwingUtilities.invokeLater(()->new ConverterFrameListener().setVisible(true));}
}
"""
named_inner = common + r"""
public class ConverterNamedInner extends JFrame {
    JTextField amount=new JTextField(10); JComboBox<String> target=new JComboBox<>(new String[]{"USD","EUR"}); JLabel result=new JLabel(" ");
    ConverterNamedInner(){super("Converter – Named Inner");setLayout(new FlowLayout());JButton b=new JButton("Convert");add(new JLabel("INR"));add(amount);add(target);add(b);add(result);Handler h=new Handler();b.addActionListener(h);amount.addActionListener(h);addWindowListener(new WindowAdapter(){public void windowClosing(WindowEvent e){dispose();}});
        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE);pack();}
    private class Handler implements ActionListener { public void actionPerformed(ActionEvent e){convert();} }
    private void convert(){try{double v=Double.parseDouble(amount.getText().trim());if(v<0||!Double.isFinite(v))throw new NumberFormatException();String c=(String)target.getSelectedItem();result.setText(new DecimalFormat("0.00").format(v*(c.equals("USD")?0.012:0.011))+" "+c);JOptionPane.showMessageDialog(this,result.getText());}catch(NumberFormatException ex){result.setText("Invalid amount");JOptionPane.showMessageDialog(this,"Invalid amount","Error",JOptionPane.ERROR_MESSAGE);}}
    public static void main(String[]a){SwingUtilities.invokeLater(()->new ConverterNamedInner().setVisible(true));}
}
"""
anonymous = common + r"""
public class ConverterAnonymous extends JFrame {
    JTextField amount=new JTextField(10); JComboBox<String> target=new JComboBox<>(new String[]{"USD","EUR"}); JLabel result=new JLabel(" ");
    ConverterAnonymous(){super("Converter – Anonymous Listener");setLayout(new FlowLayout());JButton b=new JButton("Convert");add(new JLabel("INR"));add(amount);add(target);add(b);add(result);ActionListener h=new ActionListener(){public void actionPerformed(ActionEvent e){convert();}};b.addActionListener(h);amount.addActionListener(h);addWindowListener(new WindowAdapter(){public void windowClosing(WindowEvent e){dispose();}});
        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE);pack();}
    private void convert(){try{double v=Double.parseDouble(amount.getText().trim());if(v<0||!Double.isFinite(v))throw new NumberFormatException();String c=(String)target.getSelectedItem();result.setText(new DecimalFormat("0.00").format(v*(c.equals("USD")?0.012:0.011))+" "+c);JOptionPane.showMessageDialog(this,result.getText());}catch(NumberFormatException ex){result.setText("Invalid amount");JOptionPane.showMessageDialog(this,"Invalid amount","Error",JOptionPane.ERROR_MESSAGE);}}
    public static void main(String[]a){SwingUtilities.invokeLater(()->new ConverterAnonymous().setVisible(true));}
}
"""
lambda_src = common + r"""
public class Converter extends JFrame {
    private final JTextField amount=new JTextField(10);
    private final JComboBox<String> target=new JComboBox<>(new String[]{"USD","EUR"});
    private final JLabel result=new JLabel(" ");
    private static double rate(String currency){return currency.equals("USD")?0.012:0.011;}
    public Converter(){super("Currency Converter");setLayout(new FlowLayout());JButton convertButton=new JButton("Convert");add(new JLabel("INR"));add(amount);add(target);add(convertButton);add(result);
        ActionListener convertAction=e->convert(); convertButton.addActionListener(convertAction); amount.addActionListener(convertAction);
        addWindowListener(new WindowAdapter(){public void windowClosing(WindowEvent e){dispose();}});
        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE);pack();
    }
    private void convert(){try{double value=Double.parseDouble(amount.getText().trim());if(value<0||!Double.isFinite(value))throw new NumberFormatException();String currency=(String)target.getSelectedItem();String formatted=new DecimalFormat("0.00").format(value*rate(currency));result.setText(formatted+" "+currency);JOptionPane.showMessageDialog(this,formatted+" "+currency,"Conversion Result",JOptionPane.INFORMATION_MESSAGE);}catch(NumberFormatException ex){result.setText("Invalid amount");JOptionPane.showMessageDialog(this,"Invalid amount","Input Error",JOptionPane.ERROR_MESSAGE);}}
    public static void main(String[]args){SwingUtilities.invokeLater(()->new Converter().setVisible(true));}
}
"""
(BASE/"Task05"/"ConverterFrameListener.java").write_text(frame_impl)
(BASE/"Task05"/"ConverterNamedInner.java").write_text(named_inner)
(BASE/"Task05"/"ConverterAnonymous.java").write_text(anonymous)
(BASE/"Task05"/"Converter.java").write_text(lambda_src)
(BASE/"Task05"/"comparison.txt").write_text("""Task 5.1 – Four styles
(a) Frame implements ActionListener: the frame is the listener; actionPerformed delegates to convert(). It can access fields directly but mixes view and handler responsibilities.
(b) Named inner class: a reusable named Handler class implements ActionListener and can access enclosing fields. It is clearer when handler logic is substantial.
(c) Anonymous inner class: handler is declared at registration. It can access enclosing fields but is less reusable and can become verbose.
(d) Lambda: concise functional listener for the single-method ActionListener interface. It can access enclosing fields and is ideal for short actions.
All versions register the same handler for the button and amount field (Enter key), display a dialog for success and invalid input, and use EXIT_ON_CLOSE for the frame.

Task 5.2 – Change effort
Adding Enter-key conversion requires registering the same handler on the JTextField in addition to the JButton. The frame listener and named/anonymous versions reuse their handler object; the lambda version reuses one ActionListener variable. Showing a dialog is one additional message-dialog call in convert() for each implementation.

Task 5.3 – Comparison
Readability: lambda is shortest; named inner class is clearest for complex logic; anonymous class is explicit but verbose; frame listener is simple for tiny examples but couples roles. Field access: all nested/anonymous/lambda handlers can access enclosing fields; frame listener accesses instance fields directly. Reuse: named inner class and shared Action object are easiest to reuse; lambda variable can be shared among triggers. For a large application, prefer a separate Action class or shared Action object to centralise enablement, icons, keyboard shortcuts and menu/button behaviour.

Alternative: a method reference button.addActionListener(e -> convert()) or button.addActionListener(this::convert) shares the convert method. A single AbstractAction can be attached to a button, menu item and key binding. One shared Action object can replace multiple listener instances (O(1) handler objects per action), and adding a trigger is one registration operation. Separate listeners for k triggers may use O(k) listener objects; dispatch itself is O(1) for each registered listener callback. The assignment's k-comparison estimate applies to an if/else command chain; dedicated listeners avoid scanning k commands.
""")
# headless conversion logic tests
(BASE/"Task05"/"ConverterLogic.java").write_text("""public class ConverterLogic {\n public static String convert(String text,String currency){try{double v=Double.parseDouble(text.trim());if(v<0||!Double.isFinite(v))return \"Invalid amount\";double rate=currency.equals(\"USD\")?0.012:0.011;return String.format(java.util.Locale.US,\"%.2f %s\",v*rate,currency);}catch(Exception e){return \"Invalid amount\";}}\n public static void main(String[] a){System.out.println(\"1000 INR to USD: \"+convert(\"1000\",\"USD\"));System.out.println(\"1000 INR to EUR: \"+convert(\"1000\",\"EUR\"));System.out.println(\"Invalid input: \"+convert(\"abc\",\"USD\"));}\n}\n""")
print("Task 5 four implementations written.")


Task 5 four implementations written.


## 7. Task 6 – Campus event registration Swing–JDBC application

In [7]:
task6 = r"""
import java.awt.*;
import java.awt.event.*;
import java.io.*;
import java.sql.*;
import java.util.*;
import javax.swing.*;
import javax.swing.table.DefaultTableModel;

class Registration {
    final int id; final String studentId, studentName, eventName, email;
    Registration(int id,String studentId,String studentName,String eventName,String email){this.id=id;this.studentId=studentId;this.studentName=studentName;this.eventName=eventName;this.email=email;}
}
class RegistrationDAO {
    private final String url=System.getenv().getOrDefault("EVENT_DB_URL","jdbc:mysql://localhost:3306/campus_events");
    private final String user=System.getenv().getOrDefault("EVENT_DB_USER","root");
    private final String password=System.getenv().getOrDefault("EVENT_DB_PASSWORD","");
    private Connection connect() throws SQLException {return DriverManager.getConnection(url,user,password);}
    void insert(String sid,String name,String event,String email) throws SQLException {
        String sql="INSERT INTO registrations(student_id,student_name,event_name,email) VALUES(?,?,?,?)";
        try(Connection c=connect();PreparedStatement p=c.prepareStatement(sql)){p.setString(1,sid);p.setString(2,name);p.setString(3,event);p.setString(4,email);p.executeUpdate();}
    }
    java.util.List<Registration> find(String event,int limit,int offset) throws SQLException {
        String sql=event.equals("All")?
          "SELECT id,student_id,student_name,event_name,email FROM registrations ORDER BY id LIMIT ? OFFSET ?":
          "SELECT id,student_id,student_name,event_name,email FROM registrations WHERE event_name=? ORDER BY id LIMIT ? OFFSET ?";
        java.util.List<Registration> out=new ArrayList<>();
        try(Connection c=connect();PreparedStatement p=c.prepareStatement(sql)){
            int i=1;if(!event.equals("All"))p.setString(i++,event);p.setInt(i++,limit);p.setInt(i,offset);
            try(ResultSet r=p.executeQuery()){while(r.next())out.add(new Registration(r.getInt(1),r.getString(2),r.getString(3),r.getString(4),r.getString(5)));}
        }return out;
    }
}
class RegistrationService {
    private final RegistrationDAO dao=new RegistrationDAO();
    void register(String sid,String name,String event,String email) throws Exception {
        if(sid.isBlank()||name.isBlank()||event.isBlank()||email.isBlank())throw new IllegalArgumentException("All fields are required");
        if(!email.matches("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+$"))throw new IllegalArgumentException("Enter a valid email");
        dao.insert(sid.trim(),name.trim(),event.trim(),email.trim());
    }
    java.util.List<Registration> filter(String event,int limit,int offset)throws SQLException{return dao.find(event,limit,offset);}
}
public class EventRegistrationApp extends JFrame {
    private final JTextField sid=new JTextField(10),name=new JTextField(12),email=new JTextField(16);
    private final JComboBox<String> event=new JComboBox<>(new String[]{"Cultural Fest","Music Night","Dance Competition","Tech Expo"});
    private final JComboBox<String> filter=new JComboBox<>(new String[]{"All","Cultural Fest","Music Night","Dance Competition","Tech Expo"});
    private final DefaultTableModel model=new DefaultTableModel(new Object[]{"ID","Student ID","Name","Event","Email"},0);
    private final JTable table=new JTable(model); private final JLabel status=new JLabel("Ready");
    private final RegistrationService service=new RegistrationService(); private int page=0; private static final int PAGE_SIZE=20;
    public EventRegistrationApp(){
        super("Campus Event Registration");setLayout(new BorderLayout(8,8));
        JMenuBar bar=new JMenuBar();JMenu file=new JMenu("File");JMenuItem refresh=new JMenuItem("Refresh"),export=new JMenuItem("Export CSV"),exit=new JMenuItem("Exit");file.add(refresh);file.add(export);file.addSeparator();file.add(exit);bar.add(file);setJMenuBar(bar);
        JPanel form=new JPanel(new GridLayout(2,4,6,6));form.setBorder(BorderFactory.createTitledBorder("Student Registration"));
        form.add(new JLabel("Student ID"));form.add(sid);form.add(new JLabel("Student Name"));form.add(name);form.add(new JLabel("Event"));form.add(event);form.add(new JLabel("Email"));form.add(email);
        JButton register=new JButton("Register"),filterButton=new JButton("Filter"),prev=new JButton("Previous"),next=new JButton("Next");
        JPanel tools=new JPanel();tools.add(register);tools.add(new JLabel("Filter event"));tools.add(filter);tools.add(filterButton);tools.add(prev);tools.add(next);
        JPanel north=new JPanel(new BorderLayout());north.add(form,BorderLayout.NORTH);north.add(tools,BorderLayout.SOUTH);
        add(north,BorderLayout.NORTH);add(new JScrollPane(table),BorderLayout.CENTER);add(status,BorderLayout.SOUTH);
        register.addActionListener(e->registerAsync());filterButton.addActionListener(e->{page=0;loadAsync();});refresh.addActionListener(e->loadAsync());prev.addActionListener(e->{if(page>0){page--;loadAsync();}});next.addActionListener(e->{page++;loadAsync();});exit.addActionListener(e->dispose());
        export.addActionListener(e->exportCsv());setDefaultCloseOperation(EXIT_ON_CLOSE);setSize(1000,600);setLocationRelativeTo(null);loadAsync();
    }
    private void registerAsync(){
        String a=sid.getText(),b=name.getText(),c=(String)event.getSelectedItem(),d=email.getText();
        new SwingWorker<Void,Void>(){protected Void doInBackground()throws Exception{service.register(a,b,c,d);return null;}
            protected void done(){try{get();status.setText("Registration saved");loadAsync();}catch(Exception ex){status.setText("Registration failed");JOptionPane.showMessageDialog(EventRegistrationApp.this,ex.getCause()==null?ex.getMessage():ex.getCause().getMessage(),"Registration Error",JOptionPane.ERROR_MESSAGE);}}}.execute();
    }
    private void loadAsync(){
        String selected=(String)filter.getSelectedItem();int offset=page*PAGE_SIZE;status.setText("Loading...");
        new SwingWorker<java.util.List<Registration>,Void>(){protected java.util.List<Registration> doInBackground()throws Exception{return service.filter(selected,PAGE_SIZE,offset);}
            protected void done(){try{model.setRowCount(0);for(Registration r:get())model.addRow(new Object[]{r.id,r.studentId,r.studentName,r.eventName,r.email});status.setText("Page "+(page+1)+" loaded");}catch(Exception ex){status.setText("Unable to load registrations: "+ex.getMessage());}}}.execute();
    }
    private void exportCsv(){
        JFileChooser chooser=new JFileChooser();if(chooser.showSaveDialog(this)!=JFileChooser.APPROVE_OPTION)return;
        try(PrintWriter out=new PrintWriter(chooser.getSelectedFile())){for(int r=0;r<model.getRowCount();r++){for(int c=0;c<model.getColumnCount();c++){if(c>0)out.print(",");out.print("\""+String.valueOf(model.getValueAt(r,c)).replace("\"","\"\"")+"\"");}out.println();}status.setText("CSV exported");}
        catch(IOException ex){JOptionPane.showMessageDialog(this,ex.getMessage(),"Export Error",JOptionPane.ERROR_MESSAGE);}
    }
    public static void main(String[]a){SwingUtilities.invokeLater(()->new EventRegistrationApp().setVisible(true));}
}
"""
(BASE/"Task06"/"EventRegistrationApp.java").write_text(task6)
sql = """CREATE DATABASE IF NOT EXISTS campus_events;
USE campus_events;
CREATE TABLE IF NOT EXISTS registrations (
  id INT AUTO_INCREMENT PRIMARY KEY,
  student_id VARCHAR(30) NOT NULL,
  student_name VARCHAR(100) NOT NULL,
  event_name VARCHAR(100) NOT NULL,
  email VARCHAR(150) NOT NULL,
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  CONSTRAINT uq_student_event UNIQUE (student_id, event_name)
);
"""
(BASE/"sql"/"campus_events.sql").write_text(sql)
notes="""Stage 1 – Requirements
Users: students/receptionists register students; committee members view, filter and export registrations.
Screens: registration form and committee registration table; File menu provides refresh, CSV export and exit.
Data: student ID, student name, event name, email, registration ID and creation time.
Rules: required fields; valid email; one registration per student per event; database uniqueness is authoritative; GUI remains responsive during JDBC operations.

Testable requirements
1. Blank required fields are rejected with a clear message.
2. Invalid email is rejected.
3. A valid registration is inserted and displayed.
4. A duplicate student/event pair is rejected by the database unique constraint.
5. Filter returns only the selected event.
6. Pagination requests at most 20 rows per page.
7. JDBC calls execute in SwingWorker background threads.
8. CSV export writes the currently displayed table rows.

Stage 2 – Screen/event flow
Launch → Registration form/table → Register → validation → JDBC insert in SwingWorker → success/error dialog/status → refresh current table page.
Filter selection → Filter button → paged SELECT → JTable update.
File → Export CSV → file chooser → CSV output.
File → Exit → close application.

Stage 3 – Critical decisions
Validation is in RegistrationService, not in Swing component callbacks alone. Duplicate checking is enforced by UNIQUE(student_id,event_name) in MySQL so concurrent application instances cannot bypass it. DAO owns SQL/JDBC. SwingWorker keeps insert and SELECT work off the EDT; done() updates Swing components on the EDT.

Stage 4 – Class layers
EventRegistrationApp (Swing UI) → RegistrationService (validation/use cases) → RegistrationDAO (JDBC) → MySQL registrations table.
Registration is the data model. Register handler captures form values and calls service in SwingWorker.doInBackground(); done() reports outcome and refreshes. Filter handler calls service.filter(event,20,page*20) in SwingWorker and replaces JTable rows in done().

Stage 5 – Demonstration cases
Normal: unique student ID + event + valid email → insert succeeds.
Invalid: blank ID or malformed email → IllegalArgumentException shown; no insert.
Duplicate: same student ID and event twice → MySQL unique constraint raises SQLException; error shown.
Database configuration: EVENT_DB_URL, EVENT_DB_USER, EVENT_DB_PASSWORD environment variables. MySQL Connector/J must be on the runtime classpath.

Alternative – load all 100,000 rows
Loading all rows transfers and materialises O(N) records and uses O(N) model memory; startup latency and EDT stalls can be substantial if performed on EDT. Paging 20 rows uses O(P) table memory for page size P and transfers O(P) rows per page; database still performs query work and indexes improve filtering/order. SwingWorker prevents blocking the EDT. OFFSET pagination can become slower for large offsets; keyset pagination is a scalable alternative.
"""
(BASE/"Task06"/"design_and_test_plan.txt").write_text(notes)
print("Task 6 application and SQL script written.")


Task 6 application and SQL script written.


## 8. Compile Java source files and run headless verification tests

In [8]:
# Compile all Java source files individually. GUI classes compile in Colab, but cannot open windows here.
compile_targets = [
    ("Task01","FeeWindow.java"),
    ("Task02","AppointmentDesk.java"),
    ("Task03","LibrarySearch.java"),
    ("Task04","DrawPad.java"),
    ("Task05","ConverterFrameListener.java"),
    ("Task05","ConverterNamedInner.java"),
    ("Task05","ConverterAnonymous.java"),
    ("Task05","Converter.java"),
    ("Task05","ConverterLogic.java"),
    ("Task06","EventRegistrationApp.java"),
]
compile_results=[]
for folder, source in compile_targets:
    p=subprocess.run(["javac",source],cwd=BASE/folder,capture_output=True,text=True)
    compile_results.append((folder,source,p.returncode,p.stdout,p.stderr))
    print(f"{folder}/{source}: {'COMPILED' if p.returncode==0 else 'FAILED'}")
    if p.returncode:
        print(p.stderr)

(BASE/"logs"/"compile_results.txt").write_text("\\n".join(
    f"{f}/{s}: {'COMPILED' if rc==0 else 'FAILED'}\\n{out}{err}" for f,s,rc,out,err in compile_results
))
assert all(x[2]==0 for x in compile_results), "At least one Java source did not compile."
print("\\nAll GUI source files compiled successfully.")


Task01/FeeWindow.java: COMPILED
Task02/AppointmentDesk.java: COMPILED
Task03/LibrarySearch.java: COMPILED
Task04/DrawPad.java: COMPILED
Task05/ConverterFrameListener.java: COMPILED
Task05/ConverterNamedInner.java: COMPILED
Task05/ConverterAnonymous.java: COMPILED
Task05/Converter.java: COMPILED
Task05/ConverterLogic.java: COMPILED
Task06/EventRegistrationApp.java: COMPILED
\nAll GUI source files compiled successfully.


## 9. Execute console verification tests

In [9]:
# Appointment model/service test
p=subprocess.run(["java","AppointmentDesk"],cwd=BASE/"Task02",capture_output=True,text=True)
print(p.stdout)
(BASE/"logs"/"Task02_test.txt").write_text(p.stdout+p.stderr)

# Currency conversion validation independent of GUI
p=subprocess.run(["java","ConverterLogic"],cwd=BASE/"Task05",capture_output=True,text=True)
print(p.stdout)
(BASE/"logs"/"Task05_converter_tests.txt").write_text(p.stdout+p.stderr)

# Record compile-time demonstration: @Override catches the misspelling.
p=subprocess.run(["javac","MouseTypoCompileFail.java"],cwd=BASE/"Task04",capture_output=True,text=True)
print("Expected compile failure:")
print(p.stderr)
(BASE/"logs"/"Task04_override_compile_error.txt").write_text(p.stdout+p.stderr)
assert p.returncode != 0 and ("does not override" in p.stderr or "not override" in p.stderr)

# Record test case matrix
tests = [
["Task","Test","Input/interaction","Expected result","Verification in notebook"],
["1","T1","blank registration number","validation error","design + FeeService source"],
["1","T2","non-numeric/negative amount","validation error","design + FeeService source"],
["2","T1","Cardiology department","Cardiology doctor list","AppointmentDesk console"],
["2","T2","blank patient ID","required-field error","AppointmentDesk console"],
["3","T1","search term entered","results table populated","Swing source compiled"],
["3","T2","resize 520x320, 760x480, 1100x700","centre results area resizes","desktop screenshot required"],
["4","T1","click at point","count increments and point stored","source compiled"],
["4","T2","minimise and restore","stored points redraw","source compiled; desktop screenshot required"],
["5","T1","1000 INR to USD","12.00 USD","ConverterLogic console"],
["5","T2","1000 INR to EUR","11.00 EUR","ConverterLogic console"],
["5","T3","abc amount","Invalid amount","ConverterLogic console"],
["6","T1","valid unique registration","row inserted","requires MySQL runtime"],
["6","T2","blank/invalid email","validation error","service source compiled"],
["6","T3","same student/event twice","unique constraint rejects duplicate","SQL constraint included; requires MySQL runtime"],
]
with open(BASE/"data"/"test_cases.csv","w",newline="") as f:
    w=csv.writer(f); w.writerows(tests)
print("Test case matrix saved.")


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Cardiology doctors: [Dr Rao, Dr Iyer]
Appointment confirmed: P1001 | Cardiology | Dr Rao | 10:30
Validation: Patient ID is required

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
1000 INR to USD: 12.00 USD
1000 INR to EUR: 11.00 EUR
Invalid input: Invalid amount

Expected compile failure:
MouseTypoCompileFail.java:3: error: method does not override or implement a method from a supertyp

## 10. Create the Assignment 4 source package

In [10]:
readme = """# Programming in Java – Assignment 4 (CO4)
**Name:** J. Subhaashree
**Register Number:** 192424008
**Course:** Programming in Java (CSA09)
**Course Outcome:** CO4 – Event-Driven GUI Design

## Java version
Java version used for compilation: see `logs/compile_results.txt` and the Colab runtime output. Desktop execution requires a JDK with AWT/Swing support.

## Task folders
- Task01 – Fee-payment GUI design review and repaired FeeWindow/FeeService.
- Task02 – Appointment desk event model, component hierarchy, pseudocode and appointment service.
- Task03 – Campus library search GUI and layout design.
- Task04 – DrawPad event repair, persistent painting and @Override compile-error demonstration.
- Task05 – Currency converter in frame listener, named inner class, anonymous class and lambda styles.
- Task06 – Layered Swing–JDBC event registration application and MySQL SQL script.

## Compile and run (desktop)
Run commands from the corresponding task folder:
- Task01: `javac FeeWindow.java` then `java FeeWindow`
- Task02: `javac AppointmentDesk.java` then `java AppointmentDesk`
- Task03: `javac LibrarySearch.java` then `java LibrarySearch`
- Task04: `javac DrawPad.java` then `java DrawPad`
- Task05: compile the selected file and run its class:
  - `javac ConverterFrameListener.java` / `java ConverterFrameListener`
  - `javac ConverterNamedInner.java` / `java ConverterNamedInner`
  - `javac ConverterAnonymous.java` / `java ConverterAnonymous`
  - `javac Converter.java` / `java Converter`
- Task06: configure MySQL and Connector/J, execute `sql/campus_events.sql`, set `EVENT_DB_URL`, `EVENT_DB_USER`, `EVENT_DB_PASSWORD`, then compile and run `EventRegistrationApp.java`.

## Database setup
The SQL script creates `campus_events.registrations` and a UNIQUE(student_id,event_name) constraint to prevent duplicate registration.

## Execution notes
AWT/Swing GUI source files are compiled in the notebook. Colab is headless, so interactive windows and screenshots must be captured on a desktop. Task02 service and Task05 conversion logic have console verification tests. Task04 includes the expected compile-time error caused by misspelling `mouseClicked` when @Override is used. Task06 requires a running MySQL server and MySQL Connector/J at runtime.
"""
(BASE/"README.md").write_text(readme)
(BASE/"docs"/"submission_notes.txt").write_text("""Assignment CO4 contains six task folders, source code, design explanations, pseudocode, SQL schema and console verification logs. Desktop-only interactions are identified in the test matrix and must be captured in a graphical desktop environment. The MySQL demonstration requires a configured MySQL server and Connector/J driver.""")
# Include compile results, test matrix, source files and SQL in zip.
zip_path=Path("/content/CSA09-192424008-Assignment4-Colab-Source.zip")
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in BASE.rglob("*"):
        if f.is_file() and f.suffix != ".class":
            z.write(f,Path("Assignment 4")/f.relative_to(BASE))
print("Created package:",zip_path)
print("Files:",sum(1 for f in BASE.rglob("*") if f.is_file()))
print("Package size:",zip_path.stat().st_size)


Created package: /content/CSA09-192424008-Assignment4-Colab-Source.zip
Files: 53
Package size: 22697


## 11. Download the generated source package
The ZIP contains the Java source files, task design answers, SQL schema, README, compile results and console test evidence. Desktop GUI screenshots and the MySQL runtime demonstration are captured after opening the applications on a computer with a graphical desktop and MySQL configured.